# Primary Explainable Machine-Learning Analysis

This notebook reproduces the primary analysis using performance indicators expressed per 90 minutes where appropriate.

## Analysis overview

The workflow includes:

- loading and auditing the corrected analytical dataset;
- defining the 14 predictors and binary match-success outcome;
- nested match-grouped cross-validation;
- Elastic Net, Random Forest, and XGBoost models;
- out-of-fold performance evaluation;
- match-level bootstrap confidence intervals;
- paired model comparisons;
- tournament-phase analyses;
- out-of-fold XGBoost SHAP analysis; and
- generation of reproducibility tables and figures.

The random seed is fixed at **42**.

Matches decided by penalty shootouts are classified according to the match result before the shootout. If the match was level, both teams are coded as non-wins.


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd

# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Repository paths
# Run this notebook from the repository notebooks/ directory.
# ------------------------------------------------------------

REPO_ROOT = Path.cwd().resolve().parent

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
TABLES_DIR = RESULTS_DIR
FIGURES_DIR = REPO_ROOT / "figures"

DATA_FILE = (
    DATA_DIR
    / "FIFA_WC2026_Analysis_Ready_Per90_Corrected.xlsx"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Required primary-analysis dataset was not found:\n"
        f"{DATA_FILE}\n\n"
        "Place FIFA_WC2026_Analysis_Ready_Per90_Corrected.xlsx "
        "inside the repository data/ directory. "
        "See data/README.md for details."
    )

# Load corrected primary-analysis dataset
df = pd.read_excel(DATA_FILE)

print("PRIMARY ML ANALYSIS SETUP")
print("=" * 65)
print("Repository root:", REPO_ROOT)
print("Dataset:", DATA_FILE.name)
print("Shape:", df.shape)
print("Unique matches:", df["Match_ID"].nunique())
print("Random seed:", RANDOM_STATE)

print("\nOutcome distribution:")
print(df["Win_Binary"].value_counts().sort_index())

print("\nOutput folders:")
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)


## Data and predictor audit

The following cells verify the analytical sample, outcome coding, predictor availability, missing values, and match grouping before model fitting.


In [ ]:
# Define the 14 predictors used in the primary analysis

predictors = [
    "Ranking_Difference",
    "Attempts_on_Target_per90",
    "Corner_Kicks_per90",
    "Crosses_per90",
    "Ball_Possession_%",
    "Completed_Passes_per90",
    "Completed_Line_Breaks_per90",
    "Defensive_Pressures_per90",
    "Forced_Turnovers_per90",
    "Second_Balls_per90",
    "Saves_per90",
    "Save_percentage",
    "Distance_per_90_recalculated",
    "Zone4_per90"
]

OUTCOME = "Win_Binary"
GROUP = "Match_ID"
PHASE = "Tournament_Phase"

# Verify required variables
required_columns = predictors + [OUTCOME, GROUP, PHASE]
missing_columns = [col for col in required_columns if col not in df.columns]

print("PRIMARY ANALYSIS VARIABLE CHECK")
print("=" * 65)

print("Number of predictors:", len(predictors))

print("\nPredictors:")
for i, predictor in enumerate(predictors, start=1):
    print(f"{i:02d}. {predictor}")

print("\nOutcome:", OUTCOME)
print("Grouping variable:", GROUP)
print("Phase variable:", PHASE)

print("\nMissing required columns:", missing_columns)

print("\nMissing values among predictors:")
print(df[predictors].isna().sum())

print("\nOutcome classes:")
print(df[OUTCOME].value_counts().sort_index())

print("\nTournament phases:")
print(df[PHASE].value_counts())

In [ ]:
# Create modeling objects

X = df[predictors].copy()
y = df[OUTCOME].astype(int).copy()
groups = df[GROUP].copy()
phase = df[PHASE].copy()

print("MODELING OBJECTS CREATED")
print("=" * 65)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups shape:", groups.shape)
print("Phase shape:", phase.shape)

print("\nUnique matches:", groups.nunique())

print("\nOutcome distribution:")
print(y.value_counts().sort_index())

print("\nData types of predictors:")
print(X.dtypes)

print("\nInfinite values:", np.isinf(X.select_dtypes(include=np.number)).sum().sum())
print("Missing predictor cells:", X.isna().sum().sum())

## Nested match-grouped cross-validation

Models are evaluated using nested stratified group cross-validation.

`Match_ID` is used as the grouping variable so that the two team observations from the same match are not split between training and testing sets.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Outer cross-validation
outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_audit = []

print("OUTER CROSS-VALIDATION AUDIT")
print("=" * 70)

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups=groups),
    start=1
):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    overlap = train_groups.intersection(test_groups)

    fold_audit.append({
        "Fold": fold,
        "Train_Observations": len(train_idx),
        "Test_Observations": len(test_idx),
        "Train_Matches": len(train_groups),
        "Test_Matches": len(test_groups),
        "Train_Wins": int(y.iloc[train_idx].sum()),
        "Train_NonWins": int(len(train_idx) - y.iloc[train_idx].sum()),
        "Test_Wins": int(y.iloc[test_idx].sum()),
        "Test_NonWins": int(len(test_idx) - y.iloc[test_idx].sum()),
        "Match_Overlap": len(overlap)
    })

fold_audit = pd.DataFrame(fold_audit)

display(fold_audit)

print("\nTotal test observations across folds:",
      fold_audit["Test_Observations"].sum())

print(
    "All folds have zero match overlap:",
    (fold_audit["Match_Overlap"] == 0).all()
)

In [ ]:
# Verify inner 5-fold stratified grouped CV
# using the training data from outer fold 1

first_train_idx, first_test_idx = next(
    outer_cv.split(X, y, groups=groups)
)

X_outer_train = X.iloc[first_train_idx]
y_outer_train = y.iloc[first_train_idx]
groups_outer_train = groups.iloc[first_train_idx]

inner_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_audit = []

for fold, (inner_train_idx, inner_val_idx) in enumerate(
    inner_cv.split(
        X_outer_train,
        y_outer_train,
        groups=groups_outer_train
    ),
    start=1
):
    inner_train_groups = set(
        groups_outer_train.iloc[inner_train_idx]
    )
    
    inner_val_groups = set(
        groups_outer_train.iloc[inner_val_idx]
    )

    overlap = inner_train_groups.intersection(inner_val_groups)

    inner_audit.append({
        "Inner_Fold": fold,
        "Train_Observations": len(inner_train_idx),
        "Validation_Observations": len(inner_val_idx),
        "Train_Matches": len(inner_train_groups),
        "Validation_Matches": len(inner_val_groups),
        "Train_Wins": int(y_outer_train.iloc[inner_train_idx].sum()),
        "Validation_Wins": int(y_outer_train.iloc[inner_val_idx].sum()),
        "Match_Overlap": len(overlap)
    })

inner_audit = pd.DataFrame(inner_audit)

print("INNER CROSS-VALIDATION AUDIT")
print("=" * 70)

display(inner_audit)

print(
    "\nAll inner folds have zero match overlap:",
    (inner_audit["Match_Overlap"] == 0).all()
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier


# 1. Elastic Net Logistic Regression
elastic_net = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=10000,
        random_state=RANDOM_STATE
    ))
])

elastic_net_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__l1_ratio": [0, 0.25, 0.50, 0.75, 1]
}


# 2. Random Forest
random_forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

random_forest_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 3, 5, 8],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.5, 1.0]
}


# 3. XGBoost
xgboost_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

xgboost_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.10],
    "model__max_depth": [2, 3, 4],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.70, 1.0],
    "model__colsample_bytree": [0.70, 1.0]
}


print("MODEL DEFINITIONS")
print("=" * 65)

print("Elastic Net combinations:",
      len(elastic_net_grid["model__C"]) *
      len(elastic_net_grid["model__l1_ratio"]))

print("Random Forest combinations:",
      len(random_forest_grid["model__n_estimators"]) *
      len(random_forest_grid["model__max_depth"]) *
      len(random_forest_grid["model__min_samples_split"]) *
      len(random_forest_grid["model__min_samples_leaf"]) *
      len(random_forest_grid["model__max_features"]))

print("XGBoost combinations:",
      len(xgboost_grid["model__n_estimators"]) *
      len(xgboost_grid["model__learning_rate"]) *
      len(xgboost_grid["model__max_depth"]) *
      len(xgboost_grid["model__min_child_weight"]) *
      len(xgboost_grid["model__subsample"]) *
      len(xgboost_grid["model__colsample_bytree"]))

print("\nModels defined successfully.")

## Model fitting and out-of-fold predictions

Elastic Net logistic regression, Random Forest, and XGBoost are tuned in the inner cross-validation loop and evaluated on held-out outer folds.

All reported pooled predictions are out-of-fold predictions.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
import time

models = {
    "Elastic Net": (elastic_net, elastic_net_grid),
    "Random Forest": (random_forest, random_forest_grid),
    "XGBoost": (xgboost_model, xgboost_grid)
}

# Store out-of-fold probabilities and selected parameters
oof_predictions = {
    name: np.full(len(df), np.nan)
    for name in models
}

best_parameters = []
fold_results = []

print("NESTED CROSS-VALIDATION STARTED")
print("=" * 70)

for model_name, (pipeline, param_grid) in models.items():

    print(f"\nMODEL: {model_name}")
    print("-" * 70)

    model_start = time.time()

    for outer_fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups),
        start=1
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        groups_train = groups.iloc[train_idx]

        # Independent inner CV for this outer training set
        inner_cv_fold = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        search = GridSearchCV(
            estimator=clone(pipeline),
            param_grid=param_grid,
            scoring="roc_auc",
            cv=inner_cv_fold,
            n_jobs=-1,
            refit=True
        )

        # Group information is supplied only to CV splitting
        search.fit(
            X_train,
            y_train,
            groups=groups_train
        )

        # Out-of-fold probability for positive class (Win = 1)
        test_probability = search.best_estimator_.predict_proba(X_test)[:, 1]

        oof_predictions[model_name][test_idx] = test_probability

        fold_auc = roc_auc_score(y_test, test_probability)

        fold_results.append({
            "Model": model_name,
            "Outer_Fold": outer_fold,
            "Test_Observations": len(test_idx),
            "ROC_AUC": fold_auc,
            "Best_Inner_ROC_AUC": search.best_score_
        })

        parameter_record = {
            "Model": model_name,
            "Outer_Fold": outer_fold
        }

        parameter_record.update(search.best_params_)
        best_parameters.append(parameter_record)

        print(
            f"Fold {outer_fold}/5 complete | "
            f"Test ROC-AUC = {fold_auc:.3f}"
        )

    elapsed = time.time() - model_start

    print(
        f"{model_name} complete | "
        f"Elapsed time = {elapsed / 60:.2f} minutes"
    )

print("\n" + "=" * 70)
print("NESTED CROSS-VALIDATION FINISHED")

print("\nOOF completeness:")

for model_name, predictions in oof_predictions.items():
    print(
        model_name,
        ":",
        np.sum(~np.isnan(predictions)),
        "of",
        len(predictions)
    )

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss
)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bin_edges[1:-1], right=False)

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if np.any(mask):
            observed = np.mean(y_true[mask])
            predicted = np.mean(y_prob[mask])
            weight = np.mean(mask)

            ece += weight * abs(observed - predicted)

    return ece


performance_rows = []

for model_name, probabilities in oof_predictions.items():

    predicted_class = (probabilities >= 0.50).astype(int)

    performance_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y, predicted_class),
        "Precision": precision_score(
            y, predicted_class, zero_division=0
        ),
        "Recall": recall_score(
            y, predicted_class, zero_division=0
        ),
        "F1": f1_score(
            y, predicted_class, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(y, probabilities),
        "Brier_Score": brier_score_loss(y, probabilities),
        "ECE_10bins": expected_calibration_error(
            y, probabilities, n_bins=10
        )
    })

performance_table = pd.DataFrame(performance_rows)

print("POOLED OUT-OF-FOLD MODEL PERFORMANCE")
print("=" * 75)

display(
    performance_table.round(3)
)

print("\nAll metrics calculated from 208 pooled OOF predictions.")
print("Classification threshold: 0.50")
print("Positive class: Win_Binary = 1")

In [ ]:
phase_performance_rows = []

for phase_name in ["Group Stage", "Knockout Stage"]:

    phase_mask = phase.eq(phase_name).to_numpy()
    y_phase = y.to_numpy()[phase_mask]

    for model_name, probabilities in oof_predictions.items():

        prob_phase = probabilities[phase_mask]
        pred_phase = (prob_phase >= 0.50).astype(int)

        phase_performance_rows.append({
            "Phase": phase_name,
            "Model": model_name,
            "N": len(y_phase),
            "Wins": int(y_phase.sum()),
            "Non_Wins": int(len(y_phase) - y_phase.sum()),
            "Accuracy": accuracy_score(y_phase, pred_phase),
            "Precision": precision_score(
                y_phase, pred_phase, zero_division=0
            ),
            "Recall": recall_score(
                y_phase, pred_phase, zero_division=0
            ),
            "F1": f1_score(
                y_phase, pred_phase, zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_phase, prob_phase
            ),
            "Brier_Score": brier_score_loss(
                y_phase, prob_phase
            ),
            "ECE_10bins": expected_calibration_error(
                y_phase, prob_phase, n_bins=10
            )
        })

phase_performance = pd.DataFrame(phase_performance_rows)

print("PHASE-SPECIFIC POOLED OOF PERFORMANCE")
print("=" * 80)

display(
    phase_performance.round(3)
)

## Predictive performance

This section evaluates discrimination, classification performance, probabilistic accuracy, and calibration using the pooled out-of-fold predictions.


In [ ]:
# 5,000-resample match-level bootstrap confidence intervals
# for pooled OOF model-performance metrics

N_BOOTSTRAP = 5000
bootstrap_rng = np.random.default_rng(RANDOM_STATE)

metric_functions = {
    "Accuracy": lambda yt, yp, yc: accuracy_score(yt, yc),
    "Precision": lambda yt, yp, yc: precision_score(
        yt, yc, zero_division=0
    ),
    "Recall": lambda yt, yp, yc: recall_score(
        yt, yc, zero_division=0
    ),
    "F1": lambda yt, yp, yc: f1_score(
        yt, yc, zero_division=0
    ),
    "ROC_AUC": lambda yt, yp, yc: roc_auc_score(yt, yp),
    "Brier_Score": lambda yt, yp, yc: brier_score_loss(yt, yp),
    "ECE_10bins": lambda yt, yp, yc: expected_calibration_error(
        yt, yp, n_bins=10
    )
}

# Map each Match_ID to its two row positions
unique_matches = groups.unique()

match_rows = {
    match_id: np.flatnonzero(groups.to_numpy() == match_id)
    for match_id in unique_matches
}

bootstrap_results = []

print("MATCH-LEVEL BOOTSTRAP STARTED")
print("=" * 70)
print("Bootstrap resamples:", N_BOOTSTRAP)
print("Unique matches:", len(unique_matches))

for model_name, probabilities in oof_predictions.items():

    print(f"\nProcessing: {model_name}")

    model_boot = {
        metric: []
        for metric in metric_functions
    }

    for b in range(N_BOOTSTRAP):

        sampled_matches = bootstrap_rng.choice(
            unique_matches,
            size=len(unique_matches),
            replace=True
        )

        sampled_rows = np.concatenate([
            match_rows[m] for m in sampled_matches
        ])

        y_boot = y.to_numpy()[sampled_rows]
        p_boot = probabilities[sampled_rows]
        c_boot = (p_boot >= 0.50).astype(int)

        # ROC-AUC requires both outcome classes
        if np.unique(y_boot).size < 2:
            continue

        for metric_name, metric_function in metric_functions.items():
            value = metric_function(
                y_boot,
                p_boot,
                c_boot
            )
            model_boot[metric_name].append(value)

    # Original pooled OOF estimates
    original_prob = probabilities
    original_class = (original_prob >= 0.50).astype(int)

    for metric_name, metric_function in metric_functions.items():

        estimate = metric_function(
            y.to_numpy(),
            original_prob,
            original_class
        )

        boot_values = np.asarray(model_boot[metric_name])

        lower = np.percentile(boot_values, 2.5)
        upper = np.percentile(boot_values, 97.5)

        bootstrap_results.append({
            "Model": model_name,
            "Metric": metric_name,
            "Estimate": estimate,
            "CI_Lower": lower,
            "CI_Upper": upper,
            "Valid_Bootstrap_Resamples": len(boot_values)
        })

    print(f"{model_name}: complete")

bootstrap_ci_table = pd.DataFrame(bootstrap_results)

print("\n" + "=" * 70)
print("MATCH-LEVEL BOOTSTRAP FINISHED")

display(
    bootstrap_ci_table.round(3)
)

In [ ]:
# Paired match-level bootstrap comparisons between models

from itertools import combinations

N_BOOTSTRAP = 5000
paired_rng = np.random.default_rng(RANDOM_STATE)

model_pairs = list(combinations(oof_predictions.keys(), 2))

comparison_metrics = {
    "ROC_AUC": lambda yt, p, c: roc_auc_score(yt, p),
    "Accuracy": lambda yt, p, c: accuracy_score(yt, c),
    "F1": lambda yt, p, c: f1_score(
        yt, c, zero_division=0
    ),
    "Brier_Score": lambda yt, p, c: brier_score_loss(yt, p)
}

paired_results = []

print("PAIRED MATCH-LEVEL BOOTSTRAP COMPARISON")
print("=" * 75)
print("Bootstrap resamples:", N_BOOTSTRAP)

for model_a, model_b in model_pairs:

    print(f"\nComparing: {model_a} vs {model_b}")

    bootstrap_differences = {
        metric: []
        for metric in comparison_metrics
    }

    for b in range(N_BOOTSTRAP):

        # Same sampled matches used for both models
        sampled_matches = paired_rng.choice(
            unique_matches,
            size=len(unique_matches),
            replace=True
        )

        sampled_rows = np.concatenate([
            match_rows[m] for m in sampled_matches
        ])

        y_boot = y.to_numpy()[sampled_rows]

        p_a = oof_predictions[model_a][sampled_rows]
        p_b = oof_predictions[model_b][sampled_rows]

        c_a = (p_a >= 0.50).astype(int)
        c_b = (p_b >= 0.50).astype(int)

        if np.unique(y_boot).size < 2:
            continue

        for metric_name, metric_function in comparison_metrics.items():

            metric_a = metric_function(
                y_boot, p_a, c_a
            )

            metric_b = metric_function(
                y_boot, p_b, c_b
            )

            bootstrap_differences[metric_name].append(
                metric_a - metric_b
            )

    # Original OOF differences
    for metric_name, metric_function in comparison_metrics.items():

        p_a_full = oof_predictions[model_a]
        p_b_full = oof_predictions[model_b]

        c_a_full = (p_a_full >= 0.50).astype(int)
        c_b_full = (p_b_full >= 0.50).astype(int)

        metric_a_full = metric_function(
            y.to_numpy(), p_a_full, c_a_full
        )

        metric_b_full = metric_function(
            y.to_numpy(), p_b_full, c_b_full
        )

        observed_difference = metric_a_full - metric_b_full

        differences = np.asarray(
            bootstrap_differences[metric_name]
        )

        lower = np.percentile(differences, 2.5)
        upper = np.percentile(differences, 97.5)

        paired_results.append({
            "Model_A": model_a,
            "Model_B": model_b,
            "Metric": metric_name,
            "A_minus_B": observed_difference,
            "CI_Lower": lower,
            "CI_Upper": upper,
            "CI_Excludes_Zero": bool(
                (lower > 0) or (upper < 0)
            ),
            "Valid_Bootstrap_Resamples": len(differences)
        })

    print("Complete")

paired_comparison_table = pd.DataFrame(paired_results)

print("\n" + "=" * 75)
print("PAIRED MODEL COMPARISONS FINISHED")

display(
    paired_comparison_table.round(3)
)

In [ ]:
# ============================================================
# SAVE PRIMARY ML RESULTS
# ============================================================

# Convert result lists to DataFrames if necessary
bootstrap_results_df = pd.DataFrame(bootstrap_results)

# Use the paired-comparison DataFrame already created
# Check common variable names safely
if "paired_comparison_table" in globals():
    paired_results_df = paired_comparison_table.copy()
elif "paired_results" in globals():
    paired_results_df = pd.DataFrame(paired_results)
else:
    raise NameError("Paired comparison results object was not found.")

# Output files
performance_file = TABLES_DIR / "Primary_Model_Performance.xlsx"
bootstrap_file = TABLES_DIR / "Primary_Model_Performance_Bootstrap_CI.xlsx"
paired_file = TABLES_DIR / "Paired_Model_Comparisons.xlsx"
phase_file = TABLES_DIR / "Phase_Specific_Model_Performance.xlsx"

# Save
performance_table.to_excel(
    performance_file,
    index=False
)

bootstrap_results_df.to_excel(
    bootstrap_file,
    index=False
)

paired_results_df.to_excel(
    paired_file,
    index=False
)

phase_performance.to_excel(
    phase_file,
    index=False
)

# Verification
print("PRIMARY MODEL RESULTS SAVED")
print("=" * 70)

print("Overall performance:", performance_file)
print("Bootstrap CIs:", bootstrap_file)
print("Paired comparisons:", paired_file)
print("Phase-specific performance:", phase_file)

print("\nFile checks:")
print("Overall performance:", performance_file.exists())
print("Bootstrap CIs:", bootstrap_file.exists())
print("Paired comparisons:", paired_file.exists())
print("Phase-specific performance:", phase_file.exists())

print("\nSaved table dimensions:")
print("Performance:", performance_table.shape)
print("Bootstrap:", bootstrap_results_df.shape)
print("Paired comparisons:", paired_results_df.shape)
print("Phase-specific:", phase_performance.shape)

In [ ]:
# ============================================================
# MODEL-SELECTION DOCUMENTATION
# ============================================================

selection_summary = pd.DataFrame({
    "Criterion": [
        "Overall ROC-AUC",
        "Overall Accuracy",
        "Overall F1",
        "Overall Brier Score",
        "Overall ECE",
        "Paired bootstrap evidence of superiority",
        "Selected model for SHAP"
    ],
    "Result": [
        "XGBoost = 0.887; Random Forest = 0.885; Elastic Net = 0.865",
        "XGBoost = 0.817; Random Forest = 0.798; Elastic Net = 0.788",
        "XGBoost = 0.753; Random Forest = 0.734; Elastic Net = 0.690",
        "XGBoost = 0.132; Random Forest = 0.137; Elastic Net = 0.148",
        "XGBoost = 0.046; Elastic Net = 0.067; Random Forest = 0.068",
        "No paired comparison had a 95% CI excluding zero",
        "XGBoost"
    ]
})

selection_file = TABLES_DIR / "Model_Selection_Documentation.xlsx"
selection_summary.to_excel(selection_file, index=False)

print("MODEL-SELECTION DOCUMENTATION")
print("=" * 75)

display(selection_summary)

print("\nInterpretation:")
print(
    "XGBoost was retained as the focal explainability model because it "
    "showed the strongest overall descriptive combination of discrimination, "
    "classification performance, probabilistic accuracy, and calibration. "
    "However, paired match-level bootstrap comparisons did not demonstrate "
    "statistical superiority over Elastic Net or Random Forest."
)

print("\nSaved:", selection_file)
print("File exists:", selection_file.exists())

## Match-level bootstrap inference

Confidence intervals and paired model comparisons are generated by resampling at the match level rather than the individual team-observation level.


In [ ]:
# ============================================================
# PRE-SHAP OBJECT AUDIT — CORRECTED
# ============================================================

print("PRE-SHAP OBJECT AUDIT")
print("=" * 70)

keywords = [
    "xgb", "xgboost", "fold", "oof",
    "model", "best", "outer"
]

candidate_objects = []

# Take a fixed snapshot of globals before iterating
global_snapshot = list(globals().items())

for name, obj in global_snapshot:
    if any(k in name.lower() for k in keywords):
        if not name.startswith("_"):
            
            try:
                object_length = len(obj)
            except (TypeError, AttributeError):
                object_length = ""
            
            candidate_objects.append({
                "Object_Name": name,
                "Object_Type": type(obj).__name__,
                "Length": object_length
            })

candidate_objects_df = pd.DataFrame(candidate_objects)

if len(candidate_objects_df) > 0:
    candidate_objects_df = (
        candidate_objects_df
        .sort_values("Object_Name")
        .reset_index(drop=True)
    )
    
    display(candidate_objects_df)
else:
    print("No candidate objects found.")

print("\nImportant:")
print("Do NOT restart the kernel.")
print("Do NOT rerun nested cross-validation.")
print("We are identifying the existing objects needed for OOF SHAP.")

In [ ]:
# ============================================================
# INSPECT SAVED FOLD RESULTS
# ============================================================

print("FOLD RESULTS STRUCTURE")
print("=" * 75)

print("Number of entries:", len(fold_results))
print("Container type:", type(fold_results).__name__)

for i, item in enumerate(fold_results):
    print(f"\nENTRY {i + 1}")
    print("-" * 50)
    print("Type:", type(item).__name__)

    if isinstance(item, dict):
        print("Keys:", list(item.keys()))

        # Show compact information for each stored object
        for key, value in item.items():
            if hasattr(value, "shape"):
                print(f"  {key}: {type(value).__name__}, shape={value.shape}")
            elif isinstance(value, (list, tuple, dict)):
                print(f"  {key}: {type(value).__name__}, length={len(value)}")
            else:
                text = str(value)
                if len(text) > 150:
                    text = text[:150] + "..."
                print(f"  {key}: {type(value).__name__} = {text}")

    else:
        text = str(item)
        if len(text) > 500:
            text = text[:500] + "..."
        print(text)

In [ ]:
# ============================================================
# INSPECT STORED BEST PARAMETERS
# ============================================================

print("BEST-PARAMETER AUDIT")
print("=" * 75)

print("Number of entries:", len(best_parameters))
print("Container type:", type(best_parameters).__name__)

for i, item in enumerate(best_parameters):
    print(f"\nENTRY {i + 1}")
    print("-" * 55)

    if isinstance(item, dict):
        for key, value in item.items():
            print(f"{key}: {value}")
    else:
        print("Type:", type(item).__name__)
        print(item)

## Out-of-fold XGBoost SHAP analysis

SHAP values are calculated for held-out observations from the outer cross-validation folds.

Positive SHAP values shift the XGBoost model output toward **win**, while negative SHAP values shift it toward **non-win**.

Feature color in the SHAP summary plot represents the predictor value, not the observed match outcome.


In [ ]:
# ============================================================
# RECONSTRUCT 5 OUTER-FOLD XGBOOST MODELS
# AND VERIFY AGAINST ORIGINAL OOF PREDICTIONS
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

print("RECONSTRUCTING XGBOOST OUTER-FOLD MODELS")
print("=" * 75)

# Extract the five stored XGBoost parameter sets
xgb_best_params = [
    item for item in best_parameters
    if item["Model"] == "XGBoost"
]

xgb_best_params = sorted(
    xgb_best_params,
    key=lambda x: x["Outer_Fold"]
)

# Containers
xgb_fold_models = []
xgb_fold_test_indices = []
xgb_reconstructed_oof = np.full(len(X), np.nan)

# Recreate exactly the same outer splits
outer_cv_reconstruction = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold_number, (train_idx, test_idx) in enumerate(
    outer_cv_reconstruction.split(X, y, groups),
    start=1
):

    print(f"\nFold {fold_number}/5")

    stored = xgb_best_params[fold_number - 1]

    # Remove metadata and remove "model__" prefix
    model_params = {
        key.replace("model__", ""): value
        for key, value in stored.items()
        if key.startswith("model__")
    }

    # Reconstruct pipeline
    reconstructed_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "model",
            XGBClassifier(
                **model_params,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ])

    # Fit ONLY on this outer training fold
    reconstructed_pipeline.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    # Predict ONLY held-out observations
    fold_probabilities = reconstructed_pipeline.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    xgb_reconstructed_oof[test_idx] = fold_probabilities

    # Preserve model and exact held-out indices for SHAP
    xgb_fold_models.append(reconstructed_pipeline)
    xgb_fold_test_indices.append(np.array(test_idx))

    print("Training observations:", len(train_idx))
    print("Held-out observations:", len(test_idx))


# ============================================================
# VERIFY AGAINST ORIGINAL XGBOOST OOF PREDICTIONS
# ============================================================

original_xgb_oof = np.asarray(
    oof_predictions["XGBoost"]
)

absolute_difference = np.abs(
    original_xgb_oof - xgb_reconstructed_oof
)

print("\n" + "=" * 75)
print("OOF RECONSTRUCTION VERIFICATION")
print("=" * 75)

print(
    "Complete reconstructed predictions:",
    np.isfinite(xgb_reconstructed_oof).sum(),
    "of",
    len(X)
)

print(
    "Maximum absolute probability difference:",
    absolute_difference.max()
)

print(
    "Mean absolute probability difference:",
    absolute_difference.mean()
)

print(
    "Predictions identical within tolerance 1e-10:",
    np.allclose(
        original_xgb_oof,
        xgb_reconstructed_oof,
        atol=1e-10,
        rtol=1e-10
    )
)

print("\nStored reconstructed models:", len(xgb_fold_models))
print("Stored held-out index sets:", len(xgb_fold_test_indices))

In [ ]:
# ============================================================
# OUT-OF-FOLD SHAP CALCULATION — XGBOOST
# ============================================================

import shap
import numpy as np
import pandas as pd

print("OUT-OF-FOLD SHAP CALCULATION")
print("=" * 75)

n_observations = len(X)
n_features = X.shape[1]

# Storage for all 208 observations
oof_shap_values = np.full(
    (n_observations, n_features),
    np.nan
)

# Also store the imputed feature values actually supplied to XGBoost
oof_shap_feature_values = np.full(
    (n_observations, n_features),
    np.nan
)

for fold_number, (pipeline, test_idx) in enumerate(
    zip(xgb_fold_models, xgb_fold_test_indices),
    start=1
):
    print(f"\nProcessing fold {fold_number}/5")

    # Components of fitted pipeline
    fitted_imputer = pipeline.named_steps["imputer"]
    fitted_xgb = pipeline.named_steps["model"]

    # Held-out observations only
    X_test = X.iloc[test_idx]

    # Transform using imputer fitted ONLY on outer training data
    X_test_imputed = fitted_imputer.transform(X_test)

    # Tree SHAP on held-out observations
    explainer = shap.TreeExplainer(fitted_xgb)
    fold_shap = explainer.shap_values(X_test_imputed)

    # Convert if needed
    fold_shap = np.asarray(fold_shap)

    # Store in original row positions
    oof_shap_values[test_idx, :] = fold_shap
    oof_shap_feature_values[test_idx, :] = X_test_imputed

    print("Held-out observations:", len(test_idx))
    print("SHAP shape:", fold_shap.shape)


# ============================================================
# COMPLETENESS CHECK
# ============================================================

print("\n" + "=" * 75)
print("OOF SHAP VERIFICATION")
print("=" * 75)

print("SHAP matrix shape:", oof_shap_values.shape)

print(
    "Observations with complete SHAP values:",
    np.isfinite(oof_shap_values).all(axis=1).sum(),
    "of",
    n_observations
)

print(
    "Missing SHAP cells:",
    np.isnan(oof_shap_values).sum()
)

print(
    "Infinite SHAP cells:",
    np.isinf(oof_shap_values).sum()
)


# ============================================================
# CREATE LABELED DATAFRAMES
# ============================================================

oof_shap_df = pd.DataFrame(
    oof_shap_values,
    columns=X.columns,
    index=X.index
)

oof_shap_feature_df = pd.DataFrame(
    oof_shap_feature_values,
    columns=X.columns,
    index=X.index
)

print("\nOOF SHAP calculation complete.")
print("SHAP DataFrame:", oof_shap_df.shape)
print("Feature-value DataFrame:", oof_shap_feature_df.shape)

In [ ]:
# ============================================================
# VERIFY SHAP ADDITIVITY AND DIRECTION
# ============================================================

from scipy.special import expit

shap_additivity_records = []

for fold_number, (pipeline, test_idx) in enumerate(
    zip(xgb_fold_models, xgb_fold_test_indices),
    start=1
):
    fitted_imputer = pipeline.named_steps["imputer"]
    fitted_xgb = pipeline.named_steps["model"]

    X_test_imputed = fitted_imputer.transform(
        X.iloc[test_idx]
    )

    explainer = shap.TreeExplainer(fitted_xgb)

    expected_value = np.asarray(
        explainer.expected_value
    ).reshape(-1)[0]

    # Raw model output reconstructed from SHAP
    shap_raw_output = (
        expected_value
        + oof_shap_values[test_idx, :].sum(axis=1)
    )

    # Convert raw log-odds to probability
    shap_probability = expit(shap_raw_output)

    # Actual model probability
    model_probability = pipeline.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    difference = np.abs(
        shap_probability - model_probability
    )

    shap_additivity_records.append({
        "Fold": fold_number,
        "Expected_Value_Raw": expected_value,
        "Max_Probability_Difference": difference.max(),
        "Mean_Probability_Difference": difference.mean()
    })

shap_additivity_table = pd.DataFrame(
    shap_additivity_records
)

print("SHAP ADDITIVITY VERIFICATION")
print("=" * 75)

display(
    shap_additivity_table.round(10)
)

print(
    "\nMaximum difference across all folds:",
    shap_additivity_table[
        "Max_Probability_Difference"
    ].max()
)

print("\nInterpretation:")
print(
    "Positive SHAP values increase the model output toward "
    "Win_Binary = 1 (win)."
)
print(
    "Negative SHAP values decrease the model output toward "
    "Win_Binary = 0 (non-win)."
)
print(
    "SHAP plot color represents the predictor value, "
    "not the observed outcome class."
)

In [ ]:
# ============================================================
# OVERALL AND PHASE-SPECIFIC OOF SHAP IMPORTANCE
# ============================================================

# Absolute SHAP values
abs_shap = np.abs(oof_shap_values)

# Phase masks
group_mask = (
    df["Tournament_Phase"]
    .eq("Group Stage")
    .to_numpy()
)

knockout_mask = (
    df["Tournament_Phase"]
    .eq("Knockout Stage")
    .to_numpy()
)

# Mean absolute SHAP importance
overall_importance = abs_shap.mean(axis=0)
group_importance = abs_shap[group_mask].mean(axis=0)
knockout_importance = abs_shap[knockout_mask].mean(axis=0)

shap_importance_table = pd.DataFrame({
    "Predictor": predictors,
    "Overall_MeanAbsSHAP": overall_importance,
    "Group_MeanAbsSHAP": group_importance,
    "Knockout_MeanAbsSHAP": knockout_importance
})

# Phase difference: Knockout minus Group
shap_importance_table["Knockout_minus_Group"] = (
    shap_importance_table["Knockout_MeanAbsSHAP"]
    - shap_importance_table["Group_MeanAbsSHAP"]
)

# Overall ranking
shap_importance_table["Overall_Rank"] = (
    shap_importance_table["Overall_MeanAbsSHAP"]
    .rank(method="min", ascending=False)
    .astype(int)
)

shap_importance_table = (
    shap_importance_table
    .sort_values("Overall_MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

print("OOF SHAP FEATURE IMPORTANCE")
print("=" * 90)

display(
    shap_importance_table.round(4)
)

print("\nTop 5 overall predictors:")

for i, row in shap_importance_table.head(5).iterrows():
    print(
        f"{i + 1}. {row['Predictor']} "
        f"(mean |SHAP| = {row['Overall_MeanAbsSHAP']:.4f})"
    )

In [ ]:
# ============================================================
# 5,000 MATCH-LEVEL BOOTSTRAP FOR OOF SHAP IMPORTANCE
# ============================================================

N_BOOTSTRAP = 5000
shap_rng = np.random.default_rng(RANDOM_STATE)

match_ids = df["Match_ID"].to_numpy()
phase_values = df["Tournament_Phase"].to_numpy()

unique_match_ids = pd.unique(match_ids)

match_to_rows = {
    match_id: np.flatnonzero(match_ids == match_id)
    for match_id in unique_match_ids
}

n_features = len(predictors)

overall_boot = np.full(
    (N_BOOTSTRAP, n_features),
    np.nan
)

phase_diff_boot = np.full(
    (N_BOOTSTRAP, n_features),
    np.nan
)

print("SHAP MATCH-LEVEL BOOTSTRAP STARTED")
print("=" * 75)
print("Bootstrap resamples:", N_BOOTSTRAP)
print("Unique matches:", len(unique_match_ids))

for b in range(N_BOOTSTRAP):

    sampled_matches = shap_rng.choice(
        unique_match_ids,
        size=len(unique_match_ids),
        replace=True
    )

    sampled_rows = np.concatenate([
        match_to_rows[m]
        for m in sampled_matches
    ])

    sampled_abs_shap = abs_shap[sampled_rows]
    sampled_phase = phase_values[sampled_rows]

    # Overall importance
    overall_boot[b, :] = sampled_abs_shap.mean(axis=0)

    # Phase-specific importance
    group_rows = sampled_phase == "Group Stage"
    knockout_rows = sampled_phase == "Knockout Stage"

    # Both phases should be represented
    if group_rows.any() and knockout_rows.any():

        group_mean = sampled_abs_shap[
            group_rows
        ].mean(axis=0)

        knockout_mean = sampled_abs_shap[
            knockout_rows
        ].mean(axis=0)

        phase_diff_boot[b, :] = (
            knockout_mean - group_mean
        )


# ============================================================
# 95% PERCENTILE CONFIDENCE INTERVALS
# ============================================================

overall_ci_lower = np.nanpercentile(
    overall_boot, 2.5, axis=0
)

overall_ci_upper = np.nanpercentile(
    overall_boot, 97.5, axis=0
)

phase_ci_lower = np.nanpercentile(
    phase_diff_boot, 2.5, axis=0
)

phase_ci_upper = np.nanpercentile(
    phase_diff_boot, 97.5, axis=0
)


shap_bootstrap_table = pd.DataFrame({
    "Predictor": predictors,
    "Overall_MeanAbsSHAP": overall_importance,
    "Overall_CI_Lower": overall_ci_lower,
    "Overall_CI_Upper": overall_ci_upper,
    "Group_MeanAbsSHAP": group_importance,
    "Knockout_MeanAbsSHAP": knockout_importance,
    "Knockout_minus_Group": (
        knockout_importance - group_importance
    ),
    "PhaseDiff_CI_Lower": phase_ci_lower,
    "PhaseDiff_CI_Upper": phase_ci_upper
})

shap_bootstrap_table["PhaseDiff_CI_Excludes_Zero"] = (
    (shap_bootstrap_table["PhaseDiff_CI_Lower"] > 0)
    |
    (shap_bootstrap_table["PhaseDiff_CI_Upper"] < 0)
)

shap_bootstrap_table = (
    shap_bootstrap_table
    .sort_values(
        "Overall_MeanAbsSHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nSHAP MATCH-LEVEL BOOTSTRAP FINISHED")
print("=" * 75)

display(
    shap_bootstrap_table.round(4)
)

print(
    "\nValid phase-difference bootstrap resamples:",
    np.isfinite(phase_diff_boot).all(axis=1).sum(),
    "of",
    N_BOOTSTRAP
)

In [ ]:
# ============================================================
# SAVE PRIMARY OOF SHAP RESULTS
# ============================================================

from pathlib import Path

shap_dir = RESULTS_DIR
shap_data_dir = RESULTS_DIR

shap_dir.mkdir(parents=True, exist_ok=True)
shap_data_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Save SHAP bootstrap summary table
# ------------------------------------------------------------

summary_path = (
    shap_dir /
    "Primary_XGBoost_OOF_SHAP_Importance.xlsx"
)

shap_bootstrap_table.to_excel(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 2. Build row-level OOF SHAP dataset
# ------------------------------------------------------------

row_level_shap = pd.DataFrame({
    "Match_ID": df["Match_ID"].values,
    "Team": df["Team"].values,
    "Opponent": df["Opponent"].values,
    "Tournament_Phase": df["Tournament_Phase"].values,
    "Win_Binary": y.values
})

# Add actual/imputed feature values
for predictor in predictors:
    row_level_shap[
        f"VALUE__{predictor}"
    ] = oof_shap_feature_df[predictor].values

# Add OOF SHAP values
for predictor in predictors:
    row_level_shap[
        f"SHAP__{predictor}"
    ] = oof_shap_df[predictor].values


# Add OOF predicted probability
row_level_shap[
    "OOF_XGBoost_Win_Probability"
] = original_xgb_oof


row_level_path = (
    shap_data_dir /
    "Primary_XGBoost_OOF_SHAP_Row_Level.xlsx"
)

row_level_shap.to_excel(
    row_level_path,
    index=False
)


# ------------------------------------------------------------
# 3. Save raw SHAP matrices as CSV
# ------------------------------------------------------------

shap_matrix_path = (
    shap_data_dir /
    "Primary_XGBoost_OOF_SHAP_Matrix.csv"
)

oof_shap_df.to_csv(
    shap_matrix_path,
    index=False
)

feature_matrix_path = (
    shap_data_dir /
    "Primary_XGBoost_OOF_SHAP_Feature_Values.csv"
)

oof_shap_feature_df.to_csv(
    feature_matrix_path,
    index=False
)


# ------------------------------------------------------------
# 4. Verification
# ------------------------------------------------------------

print("SHAP RESULTS SAVED")
print("=" * 75)

for path in [
    summary_path,
    row_level_path,
    shap_matrix_path,
    feature_matrix_path
]:
    print(
        path.name,
        "->",
        "SAVED" if path.exists() else "NOT FOUND"
    )

print("\nRow-level SHAP shape:", row_level_shap.shape)
print(
    "Summary table shape:",
    shap_bootstrap_table.shape
)


## SHAP interpretation and visualization

For FIFA ranking difference:

`Opponent ranking position - focal-team ranking position`

Therefore, a positive value indicates that the focal team is better ranked than its opponent.

The following cells audit SHAP direction and generate the final summary visualization.


In [ ]:
# ============================================================
# PROFESSOR COMMENT #13
# RANKING DIFFERENCE + SHAP DIRECTION AUDIT
# ============================================================

from scipy.stats import spearmanr

print("RANKING DIFFERENCE AND SHAP DIRECTION AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Verify formula again
# Ranking_Difference =
# Opponent_FIFA_Ranking - FIFA_Ranking
# ------------------------------------------------------------

expected_ranking_difference = (
    df["Opponent_FIFA_Ranking"]
    - df["FIFA_Ranking"]
)

actual_ranking_difference = df["Ranking_Difference"]

formula_difference = (
    actual_ranking_difference
    - expected_ranking_difference
)

print("\n1. FORMULA VERIFICATION")
print("-" * 50)

print(
    "Rows exactly matching formula:",
    np.isclose(
        actual_ranking_difference,
        expected_ranking_difference
    ).sum(),
    "of",
    len(df)
)

print(
    "Maximum absolute discrepancy:",
    np.abs(formula_difference).max()
)

print(
    "\nFormula:"
    "\nRanking_Difference = "
    "Opponent_FIFA_Ranking - FIFA_Ranking"
)

print(
    "\nInterpretation:"
    "\nPositive value = focal team is BETTER ranked "
    "(lower FIFA ranking number) than opponent."
)

print(
    "Negative value = focal team is WORSE ranked "
    "(higher FIFA ranking number) than opponent."
)


# ------------------------------------------------------------
# 2. Ranking Difference descriptive range
# ------------------------------------------------------------

print("\n\n2. RANKING DIFFERENCE DISTRIBUTION")
print("-" * 50)

print(
    actual_ranking_difference.describe()
)


# ------------------------------------------------------------
# 3. Signed OOF SHAP for Ranking Difference
# ------------------------------------------------------------

ranking_shap = (
    oof_shap_df["Ranking_Difference"]
    .to_numpy()
)

ranking_values = (
    oof_shap_feature_df["Ranking_Difference"]
    .to_numpy()
)

rho, p_value = spearmanr(
    ranking_values,
    ranking_shap
)

print("\n\n3. RANKING DIFFERENCE vs SIGNED OOF SHAP")
print("-" * 50)

print(
    f"Spearman correlation = {rho:.4f}"
)

print(
    f"P-value = {p_value:.6g}"
)

print(
    "Mean signed SHAP when Ranking_Difference > 0:",
    ranking_shap[ranking_values > 0].mean()
)

print(
    "Mean signed SHAP when Ranking_Difference < 0:",
    ranking_shap[ranking_values < 0].mean()
)

if np.any(ranking_values == 0):
    print(
        "Mean signed SHAP when Ranking_Difference = 0:",
        ranking_shap[ranking_values == 0].mean()
    )


# ------------------------------------------------------------
# 4. Quartile-based descriptive interpretation
# ------------------------------------------------------------

ranking_audit_df = pd.DataFrame({
    "Ranking_Difference": ranking_values,
    "Ranking_SHAP": ranking_shap
})

ranking_audit_df["Ranking_Quartile"] = pd.qcut(
    ranking_audit_df["Ranking_Difference"],
    q=4,
    duplicates="drop"
)

quartile_summary = (
    ranking_audit_df
    .groupby(
        "Ranking_Quartile",
        observed=True
    )
    .agg(
        N=("Ranking_SHAP", "size"),
        Mean_Ranking_Difference=(
            "Ranking_Difference", "mean"
        ),
        Mean_Signed_SHAP=(
            "Ranking_SHAP", "mean"
        ),
        Mean_Absolute_SHAP=(
            "Ranking_SHAP",
            lambda x: np.abs(x).mean()
        )
    )
    .reset_index()
)

print("\n\n4. QUARTILE SUMMARY")
print("-" * 50)

display(
    quartile_summary.round(4)
)


# ------------------------------------------------------------
# 5. Explicit SHAP interpretation
# ------------------------------------------------------------

print("\n5. SHAP INTERPRETATION")
print("-" * 50)

print(
    "Positive SHAP value -> pushes XGBoost output "
    "toward Win_Binary = 1 (win)."
)

print(
    "Negative SHAP value -> pushes XGBoost output "
    "toward Win_Binary = 0 (non-win)."
)

print(
    "In a SHAP beeswarm plot, COLOR represents "
    "the FEATURE VALUE, not win/non-win status."
)

print(
    "Therefore, for Ranking_Difference:"
    "\n  High/red values = larger/more positive ranking difference"
    "\n                  = focal team relatively better ranked."
    "\n  Low/blue values = smaller/more negative ranking difference"
    "\n                  = focal team relatively worse ranked."
)

In [ ]:
# ============================================================
# PRIMARY OOF SHAP BEESWARM FIGURE — XGBOOST
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

figure_dir = FIGURES_DIR
figure_dir.mkdir(parents=True, exist_ok=True)

# More publication-friendly feature labels
feature_labels = {
    "Ranking_Difference": "FIFA ranking difference",
    "Attempts_on_Target_per90": "Attempts on target per 90",
    "Corner_Kicks_per90": "Corner kicks per 90",
    "Crosses_per90": "Crosses per 90",
    "Ball_Possession_%": "Ball possession (%)",
    "Completed_Passes_per90": "Completed passes per 90",
    "Completed_Line_Breaks_per90": "Completed line breaks per 90",
    "Defensive_Pressures_per90": "Defensive pressures per 90",
    "Forced_Turnovers_per90": "Forced turnovers per 90",
    "Second_Balls_per90": "Second balls per 90",
    "Saves_per90": "Saves per 90",
    "Save_percentage": "Save percentage",
    "Distance_per_90_recalculated": "Distance covered per 90",
    "Zone4_per90": "Zone 4 low-speed sprinting per 90"
}

display_feature_names = [
    feature_labels[col] for col in X.columns
]

# Build modern SHAP Explanation object
shap_explanation = shap.Explanation(
    values=oof_shap_values,
    data=oof_shap_feature_values,
    feature_names=display_feature_names
)

# ------------------------------------------------------------
# Create beeswarm
# ------------------------------------------------------------

plt.figure(figsize=(10, 8))

shap.plots.beeswarm(
    shap_explanation,
    max_display=14,
    show=False
)

plt.xlabel(
    "SHAP value (impact on XGBoost raw model output)",
    fontsize=11
)

plt.title(
    "Out-of-Fold SHAP Summary for XGBoost",
    fontsize=13
)

plt.tight_layout()

png_path = (
    figure_dir /
    "Primary_XGBoost_OOF_SHAP_Beeswarm.png"
)

pdf_path = (
    figure_dir /
    "Primary_XGBoost_OOF_SHAP_Beeswarm.pdf"
)

plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("\nFIGURE SAVED")
print("=" * 75)
print("PNG:", png_path.exists(), png_path)
print("PDF:", pdf_path.exists(), pdf_path)

print("\nIMPORTANT FIGURE INTERPRETATION:")
print(
    "Left/negative SHAP -> pushes prediction toward non-win."
)
print(
    "Right/positive SHAP -> pushes prediction toward win."
)
print(
    "Blue = low predictor value; red = high predictor value."
)
print(
    "Color does NOT represent win/non-win outcome."
)


In [ ]:
# ============================================================
# FINAL PUBLICATION VERSION — OOF SHAP BEESWARM
# ============================================================

plt.figure(figsize=(10, 8))

shap.plots.beeswarm(
    shap_explanation,
    max_display=14,
    show=False
)

plt.axvline(
    x=0,
    linewidth=0.8,
    linestyle="--"
)

plt.xlabel(
    "SHAP value (← toward non-win | toward win →)",
    fontsize=11
)

plt.title(
    "Out-of-Fold SHAP Summary for XGBoost",
    fontsize=13
)

plt.tight_layout()

final_png = (
    figure_dir /
    "Primary_XGBoost_OOF_SHAP_Beeswarm_FINAL.png"
)

final_pdf = (
    figure_dir /
    "Primary_XGBoost_OOF_SHAP_Beeswarm_FINAL.pdf"
)

plt.savefig(
    final_png,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    final_pdf,
    bbox_inches="tight"
)

plt.show()

print("FINAL SHAP FIGURE SAVED")
print("=" * 75)
print("PNG:", final_png.exists())
print("PDF:", final_pdf.exists())

print("\nSuggested figure note:")
print(
    "SHAP values are out-of-fold explanations from the XGBoost model. "
    "Positive SHAP values shift the model output toward win and negative "
    "values toward non-win. Color represents the observed predictor value "
    "(red = high; blue = low), not match outcome. "
    "For FIFA ranking difference, positive values indicate that the focal "
    "team was better ranked than its opponent."
)